In [ ]:
import lsdb
from dask.distributed import Client
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import multiband_fit_template_numba as mbft
import nested_pandas as npd
from nested_pandas.utils import count_nested
from scipy.signal import find_peaks

import logging

## Setup and DP2 Loading

In [ ]:
client=Client(n_workers=4, memory_limit="6GB", silence_logs=logging.ERROR)

In [ ]:
cone = lsdb.ConeSearch(ra=300.0, dec=-25.0, radius_arcsec=7000.0)

dp2 = lsdb.open_catalog("/sdf/data/rubin/shared/lsdb_commissioning/hats/v30_0_6/object_collection",
                        columns=['objectId', 'coord_dec', 'coord_ra', 'objectForcedSource', 'u_psfMag', 'u_psfMagErr',
                                 'g_psfMag', 'g_psfMagErr', 'r_psfMag', 'r_psfMagErr', 'i_psfMag', 'i_psfMagErr', 'z_psfMag', 'z_psfMagErr',
                                 'y_psfMag', 'y_psfMagErr', 'ebv'],
                        search_filter=cone)

dp2

,objectId,coord_dec,coord_ra,objectForcedSource,u_psfMag,u_psfMagErr,g_psfMag,g_psfMagErr,r_psfMag,r_psfMagErr,i_psfMag,i_psfMagErr,z_psfMag,z_psfMagErr,y_psfMag,y_psfMagErr,ebv
npartitions=266,,,,,,,,,,,,,,,,,
"Order: 8, Pixel: 767837",int64[pyarrow],double[pyarrow],double[pyarrow],"nested<parentObjectId: [int64], coord_ra: [dou...",float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow]
"Order: 8, Pixel: 767838",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 8, Pixel: 780353",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 8, Pixel: 780354",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


# 1. Filter to RR Lyrae Candidate Subset

In [ ]:
# load in the fitter template
tem = mbft.load_template_dir("lsst_template")

# Define Flag Columns
nested_cols = dp2.meta.get_subcolumns("objectForcedSource")
FLAG_COLS = [col for col in nested_cols if 'Flag' in col or 'flag' in col]
FLAG_COLS

['objectForcedSource.psfFlux_flag',
 'objectForcedSource.psfDiffFlux_flag',
 'objectForcedSource.diff_PixelFlags_nodataCenter',
 'objectForcedSource.pixelFlags_bad',
 'objectForcedSource.pixelFlags_cr',
 'objectForcedSource.pixelFlags_crCenter',
 'objectForcedSource.pixelFlags_edge',
 'objectForcedSource.pixelFlags_interpolated',
 'objectForcedSource.pixelFlags_interpolatedCenter',
 'objectForcedSource.pixelFlags_nodata',
 'objectForcedSource.pixelFlags_saturated',
 'objectForcedSource.pixelFlags_saturatedCenter',
 'objectForcedSource.pixelFlags_suspect',
 'objectForcedSource.pixelFlags_suspectCenter',
 'objectForcedSource.invalidPsfFlag']

In [ ]:
# a bunch of helper functions to help us whittle down the big data set
# this one does what it says on the tin
def count_points(df, new_name='n_lc'):
    # Asked to count `lc`, this will add a column called `n_lc`
    return count_nested(df, "objectForcedSource").rename(columns={'n_objectForcedSource':new_name})

def calc_std_partition(df, band='g'):
    df = npd.NestedFrame(df)
    
    # filter to requested band (flags already removed by filter_flags_partition)
    lc = df.query(f"objectForcedSource.band == '{band}'")['objectForcedSource']
    
    if len(lc) == 0:
        return df.assign(std=np.nan)
    
    # vectorized min/max over all objects at once
    std = lc['psfMag'].groupby(level=0).std()
    
    # objects with no observations in this band get nan automatically
    std.name = 'std'
    return df.join(std)

def dust_correction_single_band(df, band):
    A_band = df['ebv'] * tem['dust'][band]
    corrected = (df[f'{band}_psfMag'] - A_band)
    df[f'{band}_psfMagExt'] = corrected
    return df

def dust_correction(df):    
    for band in tem['dust'].keys():
        df = dust_correction_single_band(df, band)
    return df

# Compiles various filtering functions into one
def filtering(df):
    # dust correction
    df = dust_correction(df)

    # color cuts
    ug_query = 'u_psfMagExt - g_psfMagExt > 0.500 and u_psfMagExt - g_psfMagExt < 1.4'
    gr_query = 'g_psfMagExt - r_psfMagExt > -0.15 and g_psfMagExt - r_psfMagExt < 0.4'
    ri_query = 'r_psfMagExt - i_psfMagExt > -0.25 and r_psfMagExt - i_psfMagExt < 0.3'
    iz_query = 'i_psfMagExt - z_psfMagExt > -0.21 and i_psfMagExt - z_psfMagExt < 0.45'
    zy_query = 'z_psfMagExt - y_psfMagExt > -0.22 and z_psfMagExt - y_psfMagExt < 0.15'
    color_query = f'{ug_query} and {gr_query} and {ri_query} and {iz_query} and {zy_query}'
    df = df.query(color_query)

    # quality flag cuts
    flag_query = " and ".join(f"{col} == False" for col in FLAG_COLS)
    df = df.query(flag_query)

    # lc length cut
    df = count_points(df).query("n_lc >= 20")

    # variability cut
    df = calc_std_partition(df).query('std >= 0.1')
    return df

In [ ]:
filtered = dp2.map_partitions(filtering)

# prune columns for downstream memory management
filtered = filtered.drop(FLAG_COLS)
filtered

,objectId,coord_dec,coord_ra,objectForcedSource,u_psfMag,u_psfMagErr,g_psfMag,g_psfMagErr,r_psfMag,r_psfMagErr,i_psfMag,i_psfMagErr,z_psfMag,z_psfMagErr,y_psfMag,y_psfMagErr,ebv,u_psfMagExt,g_psfMagExt,r_psfMagExt,i_psfMagExt,z_psfMagExt,y_psfMagExt,n_lc,std
npartitions=266,,,,,,,,,,,,,,,,,,,,,,,,,
"Order: 8, Pixel: 767837",int64[pyarrow],double[pyarrow],double[pyarrow],"nested<parentObjectId: [int64], coord_ra: [dou...",float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],float64
"Order: 8, Pixel: 767838",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 8, Pixel: 780353",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 8, Pixel: 780354",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [ ]:
%%time
# check number of surviving objects
def check_len(df):
    return len(df)
lens = filtered.map_partitions(check_len).compute()
lens["result"].sum()

CPU times: user 59.9 s, sys: 3.06 s, total: 1min 2s
Wall time: 1min 45s


np.int64(4994)

## 2. Run Fitting

In [ ]:
def template_fitting(tem, lc, print_outputs = False, fit_n = 20, coeff_n = 10, omega_n = 20, period_range=[0.2, 0.9], cols=['midpointMjdTai', 'band', 'psfMag', 'psfMagErr']):
    '''
    Returns a dictionary with coeffs (mu, d, a, phi), pests (top 3), cov (of linear params: mu (distance modulus), d (dust), a (amplitude)), sigma_P (local uncertainty of a given period), 
        like_P (global posterior likelihood uncertainty), the next best 3 periods, gen_lc (the generated light curve)
    '''
    # compute best coefficients
    omegas = np.arange(1/period_range[-1], 1/period_range[0], 0.1/omega_n) #periods from [0.2, 0.9]: frequencies from [1.1, 5.0]

    lc_bands = list(lc[cols[1]].unique())
    # print(f"template fitting lc_bands: {lc_bands}")
    
    
    rss = mbft.FitTemplate_multiband(tem, lc, omegas, NN=fit_n, use_errors=True, use_dust=False, use_band_shift=True, cols = cols)
    rss = np.array(rss)
    
    best_omega = omegas[np.argmin(rss)]
    best_pest = 1/best_omega
    
    coeffs, cov = mbft.ComputeCoeffsAndCov_multiband(tem, lc, float(best_omega), NN=coeff_n, use_errors=True, use_dust=False, cols = cols)

    # calculate error and posterior on period
    chi2_min = np.min(rss) # rss is chi2 bc it's already scaled by weights

    chi2_red = chi2_min / (len(lc.midpointMjdTai) - len(coeffs)) # length - dof
        
    periods = 1/np.array(omegas)
    rms = np.sqrt(rss / (len(lc.midpointMjdTai) - len(coeffs))) # length - dof
    mask = rss <= chi2_min + 2.3
   
    sigma_P = 0.5 * np.abs(periods[mask][0] - periods[mask][-1])

    # now get posterior likelihood to figure out how likely this is to be the right answer
    peaks, props = find_peaks(-rss, prominence=10, width=0.1)
    try:
        next_best_idx = np.argpartition(props['prominences'], -3)[-3:]
        next_best = periods[peaks[next_best_idx]]
        next_best_chi = rss[peaks[next_best_idx]]
    except:
        next_best = []

    L = np.exp(-0.5 * (rss - rss.min()))  # subtract min to prevent underflow
    norm = np.trapezoid(L, periods) 
    if norm == 0 or ~np.isfinite(norm):
        like_P = np.nan
    else:
        L /= norm # prob density
        P_mean = np.trapezoid(periods * L, periods) # mean of posterior
        P_var  = np.trapezoid((periods - P_mean)**2 * L, periods) # variance on posterior
        like_P = np.sqrt(P_var) # std/likelihood of posterior

    if print_outputs:
        print("omega_best:", best_omega)
        print(f"pest: {best_pest:0.4f} +- {sigma_P:0.6f}, global uncertainty: {like_P:0.4f}")
        print("coeffs (mu, d, a, phi):", coeffs)  # [mu, d, a, phi]
        # print("cov (mu, d, a):\n", cov)

    return dict({'coeffs':coeffs, 'p_est':best_pest, 'cov':cov, 'variance':sigma_P, 'posterior':like_P, 'next_best':next_best, 'rss':rss})

# fit_n is the number of steps used for fitting the period; coeff_n is the number of steps used when finding coefficients
# omega_n is the number of 0.1 period bins for finding the best period
def fit_stat_df(df, nested_column='objectForcedSource', fit_n = 20, coeff_n = 10, omega_n = 20, period_range=[0.2, 0.9]):
    cdf = df.copy() # for changing df

    # initialize all the new columns so it doesn't yell at me
    for col in ["fit_coeffs", "flat_chi2_red", "chi2_red", "wrms", "p_est", "p_err", "wrms_ratio"]:
        cdf[col] = np.nan

    # these ones are gonna be dictionaries (there's probably a better data structure for this)
    cdf["fit_coeffs"] = pd.Series(dtype=object)
    cdf["flat_chi2_red"] = pd.Series(dtype=object)
    cdf["chi2_red"] = pd.Series(dtype=object)

    # find a template fit to each object in the table, save the results
    for idx, row in cdf.iterrows(): 
        lc_raw = row[nested_column].dropna(subset=['psfMag', 'psfMagErr', 'midpointMjdTai'])
        flag_cols = [col for col in lc_raw.columns if 'flag' in col.lower()]
        lc_flag = lc_raw[~lc_raw[flag_cols].any(axis=1)]
    
        cols=['midpointMjdTai', 'band', 'psfMag', 'psfMagErr']
        band_counts = lc_flag.groupby(cols[1])[cols[2]].count()
        valid_bands = band_counts[band_counts >= 3].index.tolist()
        
        # filter lc_sorted to only valid bands
        lc = lc_flag[lc_flag[cols[1]].isin(valid_bands)].reset_index(drop=True)
        lc_bands = list(lc[cols[1]].sort_values(kind="stable").unique())
        # print(f"fit stat df lc_bands: {lc_bands}")
        if len(lc_bands) <= 1:
            print(f"{row.name} doesn't have enough bands for fitting. Skipping...")
            continue
        if len(lc) < 10:
            print(f"{row.name} only has {len(lc)} points and is unsuitable for fitting. Skipping...")
            continue
        pipeline_output = template_fitting(tem, lc, fit_n = fit_n, coeff_n = coeff_n, omega_n = omega_n, period_range=[0.2, 0.9], print_outputs=False, cols=cols)
        
        coeffs = pipeline_output['coeffs']
        pest = pipeline_output['p_est']
        tem_plot = mbft.reorder_template_for_lc(tem, lc_bands)
        gamma = tem_plot['templates']
        t = tem_plot['temp_time']
        
        abs_mag_est = tem_plot['abs_mag'](pest, tem_plot)[0]
        chi2_total = 0
        flat_chi2_total = 0
        chi2_dict = dict({})
        flat_chi2_dict = dict({})
        wrms_band = []
        scatter_ratios = []
        coeffs_out = dict({'mu': coeffs[0], 'd': coeffs[1], 'a':coeffs[2], 'phi':coeffs[3]})
        for i, band in enumerate(lc_bands):
            
            gamma_band = gamma[i, :]
            
            offset = coeffs[4 + i] if i < len(lc_bands)-1 else 0.0
            coeffs_out[band] = offset
            
            m_est=coeffs[0] + abs_mag_est[i] + coeffs[2]*gamma_band + offset
    
            band_lc = lc[lc['band'] == band]
            
            minterp = np.interp(band_lc.midpointMjdTai%pest/pest, t, m_est)
        
            model_err = tem['model_error']['g'] # same for all bands
    
            err_total = np.sqrt(band_lc.psfMagErr**2 + model_err**2)
    
            chi2 = np.sum(((band_lc.psfMag - minterp) ** 2) / err_total**2)
            chi2_total += chi2
    
            chi2_dict[band] = chi2
    
            weights = 1/err_total**2
    
            flat = np.average(band_lc.psfMag, weights=weights)
            flat_chi2 = np.sum(weights * (band_lc.psfMag - flat)**2)
    
            flat_chi2_dict[band] = flat_chi2
            flat_chi2_total += flat_chi2
    
            wrms_template = np.sqrt(np.average((band_lc.psfMag - minterp) ** 2, weights=weights))
            wrms_band.append(wrms_template)
            wrms_flat = np.sqrt(np.average((band_lc.psfMag - flat)**2, weights=weights))
            
            # Ratio: < 1 means template is doing something useful
            scatter_ratio = wrms_template / wrms_flat
            scatter_ratios.append(scatter_ratio)
            
        dof = len(lc.psfMag) - len(coeffs)
        flat_dof = len(lc.psfMag) - len(lc_bands)
        chi2_red = chi2_total/dof
        flat_chi2_red = flat_chi2_total/flat_dof
    
        
        # Weighted by number of points per band
        band_sizes = [len(lc[lc['band']==b]) for b in lc_bands]
        scatter_ratio_global = np.average(scatter_ratios, weights=band_sizes)
    
        wrms = np.average(np.array(wrms_band), weights=band_sizes)
    
        chi2_dict['total'] = chi2_red
        flat_chi2_dict['total'] = flat_chi2_red
        
        cdf.at[row.name, "chi2_red"] = chi2_dict
        cdf.at[row.name, "flat_chi2_red"] = flat_chi2_dict
        cdf.loc[row.name, "wrms"] = wrms
        cdf.loc[row.name, "p_est"] = pest
        cdf.loc[row.name, "p_err"] = pipeline_output['posterior']
        cdf.loc[row.name, "wrms_ratio"] = scatter_ratio_global
        cdf.at[row.name, "fit_coeffs"] = coeffs_out
    return cdf

In [ ]:
meta_finder = filtered.head(5)
meta = fit_stat_df(meta_finder)
meta

objectId  coord_dec    coord_ra  \
_healpix_29                                                      
3376984963737613727  756398235370674934 -26.890426  300.468557   
3376985823765776980  756398304090130592 -26.919674  300.284948   
3376985929468093244  756398304090151096 -26.908395  300.321402   
3376986016004562941  756398304090151247 -26.906101  300.298303   
3376986084556872796  756398304090151886 -26.886131  300.318667   

                                                    objectForcedSource  \
_healpix_29                                                              
3376984963737613727  [{parentObjectId: 0, coord_ra: 300.468557, coo...   
3376985823765776980  [{parentObjectId: 0, coord_ra: 300.284948, coo...   
3376985929468093244  [{parentObjectId: 0, coord_ra: 300.321402, coo...   
3376986016004562941  [{parentObjectId: 0, coord_ra: 300.298303, coo...   
3376986084556872796  [{parentObjectId: 0, coord_ra: 300.318667, coo...   

                      u_psfMag  u_psfMagErr   g_psfMag  g_psfMagErr  \
_healpix_29                                                           
3376984963737613727  24.686102     0.490821  23.179617     0.091978   
3376985823765776980   25.44828     1.692292  24.019032     0.109384   
3376985929468093244  21.546488     0.033452  20.622412     0.006356   
3376986016004562941  24.159338     0.315779  22.888605     0.039734   
3376986084556872796  21.804214     0.038755  20.917109     0.007472   

                      r_psfMag  r_psfMagErr  ...  y_psfMagExt  n_lc       std  \
_healpix_29                                  ...                                
3376984963737613727  22.777035     0.046577  ...    21.895801    44  0.119607   
3376985823765776980  23.976486     0.106029  ...    23.016531    46  0.379082   
3376985929468093244  20.195126     0.004986  ...    19.805459    43  0.127246   
3376986016004562941  22.438261     0.026403  ...    22.207712    42  0.178257   
3376986084556872796  20.655981      0.00632  ...    20.033132    44  0.204351   

                                                            fit_coeffs  \
_healpix_29                                                              
3376984963737613727  {'mu': 21.260865319830963, 'd': 0.0, 'a': 0.43...   
3376985823765776980  {'mu': 22.36771348610685, 'd': 0.0, 'a': 1.148...   
3376985929468093244  {'mu': 18.974797102764896, 'd': 0.0, 'a': 0.42...   
3376986016004562941  {'mu': 21.125536439437386, 'd': 0.0, 'a': 0.38...   
3376986084556872796  {'mu': 19.445509754086682, 'd': 0.0, 'a': 0.04...   

                                                         flat_chi2_red  \
_healpix_29                                                              
3376984963737613727  {'g': 2.7474113, 'i': 6.8167686, 'r': 3.241046...   
3376985823765776980  {'g': 4.359553, 'i': 1.5291018, 'r': 6.860862,...   
3376985929468093244  {'g': 20.16275, 'i': 117.16205, 'r': 10.13095,...   
3376986016004562941  {'g': 3.4881477, 'i': 14.726497, 'r': 3.462967...   
3376986084556872796  {'g': 50.890762, 'i': 354.4947, 'r': 51.82713,...   

                                                              chi2_red  \
_healpix_29                                                              
3376984963737613727  {'g': 0.5400044166461889, 'i': 5.8130132979731...   
3376985823765776980  {'g': 10.727142097968134, 'i': 5.5569950528975...   
3376985929468093244  {'g': 83.38328909856425, 'i': 160.437764028659...   
3376986016004562941  {'g': 7.766051324223501, 'i': 10.6757611016688...   
3376986084556872796  {'g': 52.74280189044326, 'i': 366.256443599338...   

                         wrms     p_est     p_err  wrms_ratio  
_healpix_29                                                    
3376984963737613727  0.117370  0.242653  0.203404    0.805481  
3376985823765776980  0.332882  0.433630  0.192260    1.374977  
3376985929468093244  0.183647  0.322986  0.013356    1.539267  
3376986016004562941  0.144981  0.222167  0.207037    1.107381  
3376986084556872796  0.250199  0.346487  0

In [ ]:
fitted = filtered.map_partitions(fit_stat_df, meta=meta.head(0))
fitted

,objectId,coord_dec,coord_ra,objectForcedSource,u_psfMag,u_psfMagErr,g_psfMag,g_psfMagErr,r_psfMag,r_psfMagErr,i_psfMag,i_psfMagErr,z_psfMag,z_psfMagErr,y_psfMag,y_psfMagErr,ebv,u_psfMagExt,g_psfMagExt,r_psfMagExt,i_psfMagExt,z_psfMagExt,y_psfMagExt,n_lc,std,fit_coeffs,flat_chi2_red,chi2_red,wrms,p_est,p_err,wrms_ratio
npartitions=266,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Order: 8, Pixel: 767837",int64[pyarrow],double[pyarrow],double[pyarrow],"nested<parentObjectId: [int64], coord_ra: [dou...",float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],float[pyarrow],object,object,object,float64,float64,float64,float64
"Order: 8, Pixel: 767838",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 8, Pixel: 780353",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 8, Pixel: 780354",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


# 3. Write out Results

In [ ]:
# Save to catalog
fitted.write_catalog("../dp2_fitted_small")

RuntimeError: AppendRowGroups requires equal schemas.
This schema has 65 columns, other has 68

In [ ]:
# To verify locally -- watch out for results that are too large
result = fitted.compute()
result

/home/b/brantd/lsdb/src/lsdb/catalog/dataset/healpix_dataset.py:605: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(result)


objectId  coord_dec    coord_ra  \
_healpix_29                                                      
3376984963737613727  756398235370674934 -26.890426  300.468557   
3376985823765776980  756398304090130592 -26.919674  300.284948   
...                                 ...        ...         ...   
3425948724229215012  760957703932627533  -23.33831  301.029913   
3425949069530336490  760957703932627460 -23.333927  300.986155   

                                                    objectForcedSource  \
_healpix_29                                                              
3376984963737613727  [{parentObjectId: 0, coord_ra: 300.468557, coo...   
3376985823765776980  [{parentObjectId: 0, coord_ra: 300.284948, coo...   
...                                                                ...   
3425948724229215012  [{parentObjectId: 0, coord_ra: 301.029913, coo...   
3425949069530336490  [{parentObjectId: 0, coord_ra: 300.986155, coo...   

                      u_psfMag  u_psfMagErr   g_psfMag  g_psfMagErr  \
_healpix_29                                                           
3376984963737613727  24.686102     0.490821  23.179617     0.091978   
3376985823765776980   25.44828     1.692292  24.019032     0.109384   
...                        ...          ...        ...          ...   
3425948724229215012  23.825787     0.208047  22.340546     0.024979   
3425949069530336490  23.600267     0.212599  22.312449     0.019727   

                      r_psfMag  r_psfMagErr  ...  y_psfMagExt  n_lc       std  \
_healpix_29                                  ...                                
3376984963737613727  22.777035     0.046577  ...    21.895801    44  0.119607   
3376985823765776980  23.976486     0.106029  ...    23.016531    46  0.379082   
...                        ...          ...  ...          ...   ...       ...   
3425948724229215012  21.935114     0.017043  ...    21.170692    54  0.112501   
3425949069530336490  22.021017       0.0159  ...     21.22229    41  0.727661   

                                                            fit_coeffs  \
_healpix_29                                                              
3376984963737613727  {'mu': 21.260936609767132, 'd': 0.0, 'a': 0.43...   
3376985823765776980  {'mu': 22.373082948060034, 'd': 0.0, 'a': 1.13...   
...                                                                ...   
3425948724229215012  {'mu': 20.768871334449337, 'd': 0.0, 'a': 0.60...   
3425949069530336490  {'mu': 19.595412651993772, 'd': 0.0, 'a': 2.57...   

                                                         flat_chi2_red  \
_healpix_29                                                              
3376984963737613727  {'g': 2.7474113, 'i': 6.8167686, 'r': 3.241046...   
3376985823765776980  {'g': 4.359553, 'i': 1.5291018, 'r': 6.860862,...   
...                                                                ...   
3425948724229215012  {'g': 12.563577, 'i': 74.78544, 'r': 11.24058,...   
3425949069530336490  {'g': 690.12494, 'r': 672.4842, 'y': 921.943, ...   

                                                              chi2_red  \
_healpix_29                                                              
3376984963737613727  {'g': 0.5373455829353749, 'i': 5.8157858735079...   
3376985823765776980  {'g': 11.411292355110158, 'i': 5.4795325548969...   
...                                                                ...   
3425948724229215012  {'g': 3.0001823185523064, 'i': 41.949530230371...   
3425949069530336490  {'g': 258.12578987240977, 'r': 809.09930746297...   

                         wrms     p_est     p_err  wrms_ratio  
_healpix_29                                                    
3376984963737613727  0.117353  0.242653  0.204111    0.805255  
3376985823765776980  0.334099  0.433630  0.190489    1.378983  
...                       ...       ...       ...         ...  
3425948724229215012  0.133156  0.484001  0.052209    0.841717  
3425949069530336490  0.588090  0.469239  0